# Open coagulation, conservative engine, constant kernel

Пишет `runs/open_coagulation_conservative_constant.npz`; разбирает его
`Analysis_open_coagulation_conservative_constant.ipynb`.

**Что здесь нового против геометрической run-тетради.** Она сидела на
`BF_v_no_resampling_v4` — холодном движке без перезапуска, — поэтому прогон был
один: запустил, дождался, чем кончится. Здесь стоит `BF_warm_start_v5`, и
появляется то, чего у коагуляции не было: **чекпойнт**. Рабочий порядок —
`START="cold", SAVE_STATE=True` → `AN.stationarity` → не сошлось → `START="state",
SAVE_STATE=True` → ещё раз → сошлось → `START="state", SAVE_STATE=False` и мерить.
Перезапуски составляются: `t_origin` копит время всех предыдущих ног, и возрасты
переходят через шов отрицательным временем, поэтому $\tau$ **не обрезается длиной
измерительного прогона** — а возраст у коагуляции и есть основная наблюдаемая.

**Синтетического посева здесь нет и не будет.** У фрагментации он портит только
поколения. У коагуляции наблюдаемая — возраст: изохроны, $\langle m\rangle(\tau)$,
показатель $b$. У посеянного тела возраста не существует, его пришлось бы выдумать,
и выдуманный возраст попал бы прямо в измеряемую величину.

# Majorant\_v2 — open cascades on the conservative engine

This notebook runs the open coagulation configuration on
`BF_warm_start_v5.py`. One simulated particle stands for exactly one physical
particle for the whole run, so the total number and the total mass are conserved to
machine precision and the reported `mass_drift` is identically zero; anything else is
a bug rather than a fluctuation. That is what *conservative* means in the names of
the files this notebook writes.

An open system has a source and an absorbing boundary, so a genuine steady state
exists and the spectrum to measure is the **instantaneous** one rather than a
superposition over time. $\Delta t$ never enters the estimator, which is why the
closed notebook has to agonise over the snapshot cadence and this one does not — and
which is also why every snapshot taken after the cascade reached the sink is an
independent draw from the same distribution, so they are averaged rather than thrown
away. That average costs nothing and is worth a factor $\sqrt{K}$ in noise. The theory
under test is

$$\alpha = -\frac{3+\lambda}{2}, \qquad b = \frac{2}{1-\lambda},$$

with $\lambda$ the homogeneity degree of the kernel, $\alpha$ the slope of $dN/dm$
across the inertial range and $b$ the growth exponent read from the isochrones.
Coagulation injects monomers at $m_{\rm inj}$ and absorbs products above
$m_{\rm sink}$; fragmentation injects large bodies at $m_{\rm inj}$ and absorbs
fragments below $m_{\rm sink}$. The two share $\alpha$ and $b$ exactly, and only the
direction of the drift changes.

Spectra are drawn compensated, as $m^2\,dN/dm$ — the mass per logarithmic mass
interval, whose slope is $\alpha+2$. Every fit is performed on the raw $dN/dm$ and
the compensation is applied at draw time only, so the reported index is never the
compensated one.

Each successful run is written into `runs/` by `add_last_run`, keeping the spectra,
the isochrone histogram and all the parameters but discarding the per-particle
arrays. The analysis notebooks in that folder rebuild every figure from those files,
so a plot can be changed without paying for the simulation again.


## How the index is measured

A measured spectrum is a power law only *between* the two characteristic masses of
the problem. Outside that band it bends — at the injection scale because of the
source, at the sink because of truncation and vanishing statistics — and a single fit
across the whole array averages the plateau together with both bends, returning a
number that describes neither. What is measured here instead is the plateau in the
local slope,

$$\Gamma(m)=\frac{d\log F}{d\log m},$$

computed by least squares on a sliding window. `BF.find_inertial_range` returns the
longest contiguous stretch over which $\Gamma$ stays flat within a tolerance,
together with that stretch's mean, its scatter and its width in decades. The mean is
the measured $\alpha$: it is what enters the summary table and it is the line drawn
over the spectrum. When it comes back as `nan` the run simply has no inertial range
yet, which is the correct answer rather than a failure. What it is computed on is
`steady_spectrum`, the mean of the snapshots taken after the sink gate opened, not a
single snapshot.

Running alongside it is the a-priori guard band, half a decade stripped from each end
of the interval between $m_{\rm inj}$ and $m_{\rm sink}$, chosen before anyone looks
at the data. The two must agree; where they do not, the cascade has not developed
over the range that was assumed. The width in decades is printed next to every index
because a plateau narrower than roughly one decade is not a power law however tight
its error bar looks, and because the plateau finder can only test whether the curve
you computed has a scaling region — not whether that curve is the right quantity to
have computed.


In [ ]:
import os, importlib, inspect, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
#  ДВИЖОК.  v5, а не v4 -- ради ОДНОЙ вещи: state_of() и ic={'state': ...}.
#  Всё остальное v5 делает так же, изохроны в нём те же самые (iso_counts,
#  iso_dndm, iso_age_edges присутствуют в обоих движках слово в слово).
import BF_warm_start_v5 as BF
import BF_analysis as AN
importlib.reload(BF); importlib.reload(AN)   # кэш модуля -- молча другой ответ, не ошибка

# ============================================================================
#  С ЧЕГО НАЧИНАТЬ.  Два варианта, третьего (синтетический посев) здесь нет.
# ============================================================================
#   "cold"    дельта при m_inj.  У каждой частицы возраст честный с первого
#             события, потому что все вошли одинаково.  Цена -- надо дождаться
#             стационара, а это НЕ одна промывка коробки.
#   "state"   ПЕРЕЗАПУСК с сохранённого состояния.  Спектр правильный с t = 0,
#             история честна у всех, и возрасты переносятся через шов
#             отрицательными inj_time -- поэтому tau НЕ обрезается длиной
#             прогона.  Именно это у коагуляции и стоило завести.
#
#   Посева ("seeded") нет НАМЕРЕННО.  Наблюдаемая коагуляции -- возраст; у
#   синтетически посеянного тела возраста нет, и любой приписанный ему возраст
#   попадёт прямо в <m>(tau) и в b.
#
#  РАБОЧИЙ ПОРЯДОК.  Не угадывать длину заранее, а идти прогонами:
#     START="cold",  SAVE_STATE=True  -> прогон -> AN.stationarity() -> не сошлось?
#     START="state", SAVE_STATE=True  -> ещё прогон -> проверить -> ... -> сошлось
#     START="state", SAVE_STATE=False -> измерительный прогон
#  t_origin копит время всех ног, так что «ещё 0.7 промывки» переводится в
#  FLUSHES без пересчёта.
START      = "cold"     # <<-- "cold" | "state"
SAVE_STATE = True       # <<-- сохранить состояние частиц в конце (файл ~100+ МБ)
STATE_FILE = "runs/state_open_coagulation_conservative_constant.npz"

if START not in ("cold", "state"):
    raise ValueError("START = 'cold' | 'state'")
if START == "state" and not hasattr(BF, "state_of"):
    raise ValueError("движок на пути не умеет перезапуск с состояния (нужен v5)")

plt.rcParams.update({"figure.dpi":110, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.25, "figure.figsize":(9,3.2)})

KERNEL = BF.kernel_constant          # lambda = 0 (постоянное сечение)
LAM    = BF.KERNEL_LAMBDA[KERNEL.__name__]
#  СЕТКА.  Геометрическая тетрадь брала 10**arange(-4, 12, 0.1) -- 160 бинов, из
#  которых при m_inj = 1 и m_sink = 1e6 работают меньше восьмидесяти.  Верх нужен
#  лишь настолько, чтобы вместить продукт слияния ДВУХ тел у самого стока (до 2
#  m_sink), низ -- чтобы вместить инжекцию.  Каждый лишний бин оплачивается в
#  каждой векторной операции по O(B) и во всей матрице мажоранты.
EDGES  = 10.0 ** np.arange(-1, 6.55, 0.1)

_par = inspect.signature(BF.simulate).parameters
assert "stop_sink_events" in _par, "движок на пути -- не BF_warm_start_v5.py"
assert "track_generations" in _par, "движок на пути не несёт поколений (нужен v5)"

print("engine =", BF.__name__, "| старт:", START, "| чекпойнт:", SAVE_STATE)
print("kernel =", KERNEL.__name__, " lambda =", LAM)
PR = BF.predict("open", LAM)
print("  open   beta = %.4g   b = %.4g   alpha = %.4g" % (PR["beta"], PR["b"], PR["alpha"]))
print("  grid   %.3g .. %.3g  (%d bins of %.2f dex)"
      % (EDGES[0], EDGES[-1], EDGES.size-1, np.log10(EDGES[1]/EDGES[0])))

# Age bins must be chosen from b, not copied from the lambda = 0 notebook.  One age
# bin of width D_tau dex covers b * D_tau dex of MASS, so at b = 6 the old 0.15 dex
# step jumps 0.9 dex of mass per bin and the growth-law mask catches two bins.
# Fix the MASS resolution instead and let the age step follow.
#  При lambda = 0 теория даёт b = 2, значит шаг по возрасту 0.15 dex -- втрое
#  крупнее геометрических 0.05.  Изохрон меньше, но каждая толще, и статистика на
#  бин выше ровно во столько же раз.
AGE_STEP = 0.30 / PR["b"]          # 0.15 при b = 2
print("  ages   %.3f dex per bin  ->  %.2f dex of mass per bin" % (AGE_STEP, 0.30))

RESULTS = {}   # collected for the summary table at the end


In [ ]:
# ----------------------------------------------------------------------
#  The measurement layer
# ----------------------------------------------------------------------
#  Everything that turns arrays into a number now lives in BF_analysis.py, next to
#  this notebook: the estimator and its error bars, the plateau finders, the
#  isochrone machinery and the saver.  Keeping it there rather than in a cell means
#  the run notebooks and the six analysis notebooks cannot drift apart, and that a
#  change to the estimator is one edit rather than nine.
#
#  The names are aliased below so the figure cells further down read exactly as
#  before.

import BF_analysis as AN

anchor_amplitude = AN.anchor_amplitude
pick_isochrones  = AN.pick_isochrones
draw_isochrones  = AN.draw_isochrones
compensated_ylim = AN.compensated_ylim
add_last_run     = AN.add_last_run


def steady_spectrum(run, frac=0.5):
    """
    (F, K): the averaged post-gate spectrum and how many snapshots went into it.

    AN.steady_spectrum returns the full dict -- F, the empirical and Poisson error
    bars, the raw counts and K_eff.  This shim keeps the two-value call used below;
    call AN.steady_spectrum directly when the error bars are wanted.
    """
    s = AN.steady_spectrum(run, frac=frac)
    return s["F"], s["K"]


def check_grid(*masses):
    """Refuse to start if a characteristic mass falls outside EDGES."""
    return AN.check_grid(EDGES, *masses)


---
## Open system with coagulation

Monomers are injected at $m_{\rm inj}$ at a constant rate and products crossing
$m_{\rm sink}$ are removed. The injection rate follows from the balance of particle
number in the steady state, where the accepted event rate must equal the injection
rate:

$$q = \frac{\langle K\rangle\,N_{\rm ss}^{2}}{2V}.$$

Writing $q=N^2/2$, as the original four-case notebook did, is that same expression at
$\langle K\rangle = 1$ — true for the constant kernel and for nothing else. Because
the population is dominated by the injection scale whenever $\alpha<-1$, most pairs
are $(m_{\rm inj},m_{\rm inj})$ and $\langle K\rangle$ is of order
$K(m_{\rm inj},m_{\rm inj})$, with heavier partners raising it by roughly a factor
two. `live` therefore settles somewhat below the nominal target, and that is fine;
what matters is that it settles at all and that $M_{\rm out}/M_{\rm in}$ heads
towards one.

Nothing in an open run may be gated on an absolute time. The clock advances by
$\Delta t = 2V/(wR)$ with $R\propto N^2$, so after $E$ trials

$$t \simeq \frac{2E}{N^{2}},$$

and raising $N$ at a fixed event budget makes the run *shorter* in physical time
rather than longer. A gate expressed as a time and tuned at one $N$ is never crossed
at ten times that $N$. Both gates used below are therefore counted in sink
absorptions: the isochrones begin accumulating once `iso_start_sink` particles have
crossed the entire inertial range, and the run ends after `stop_sink_events` of them.
Each is a statement about the physics and needs no retuning when $N$, the mass range
or the kernel changes. A ceiling on `max_events` is kept alongside them, not as the
criterion but as the guarantee that a mistuned $q$ cannot cost an afternoon;
`stop_reason` afterwards says which of the two actually bit.

**Стационар и как его дождаться.** Оба привычных критерия — $M_{\rm out}/M_{\rm in}$
и $dn_{\rm out}/d\,{\rm ev}$ — садятся на единицу задолго до того, как садится
**форма**: число живёт у стока и релаксирует за $t_{\rm turn}$, а форма живёт наверху
и релаксирует за $\tau_{\rm res}$, в $\langle m\rangle/m_{\rm sink}$ раз дольше.
Судить поэтому по $\langle m\rangle$ и $\alpha$, сравненным между половинами хвоста:
это и делает `AN.stationarity`, и её вердикт печатается под прогоном. Не сошлось —
`START="state"`, ещё прогон с того же чекпойнта; сошлось — `SAVE_STATE=False` и
измерение. Отказывать в сохранении нестационарного прогона было бы ошибкой:
итеративный порядок работы на этом и держится.

The isochrone age bins are derived from $b$ rather than copied from a run at another
kernel. One age bin of width $\Delta_\tau$ dex spans $b\,\Delta_\tau$ dex of mass, so
the mass resolution is fixed at $0.3$ dex and the age step follows as
$\Delta_\tau = 0.3/b$.


In [ ]:
# OPEN system: the steady state is the LAST snapshot, dt never enters the estimator,
# so snapshotting by events is perfectly fine here.
#
# The isochrone gate is PHYSICAL, not temporal: bin ages only after the sink has swallowed
# N_OUT particles.  In v2 that gate is armed by default (10) for BOTH processes; it is passed
# explicitly below only to keep the number visible next to the budget it implies.

import time

N_SS    = 1e6
M_INJ   = 1.0
M_SINK  = 1e6
R       = M_SINK / M_INJ
N_OUT   = 10        # поглощений до начала набора изохрон (ворота)
N_STOP  = 100       # поглощений до ОСТАНОВКИ измерительного прогона
FLUSHES = 2.0       # длина СТРОЯЩЕГО прогона, в промывках коробки
HOURS   = 3.0       # потолок по времени, чтобы неверное q не стоило вечера
SEED    = 3
#  ПОКОЛЕНИЯ ВКЛЮЧЕНЫ НЕ РАДИ ПОКОЛЕНИЙ.  В коагуляции gen -- это счётчик
#  слияний, и сам по себе он тут не измеряется.  Но final_gen и final_gen_time
#  движок кладёт в результат ТОЛЬКО внутри ветки track_generations, а state_of()
#  без них отказывается собирать чекпойнт.  Один int32 на частицу -- цена
#  возможности продолжить прогон.
GEN_MAX = 64
check_grid(M_INJ, M_SINK)

#  МАССА КОРОБКИ и ПРОМЫВКА.  Промывка -- сколько СОБЫТИЙ нужно, чтобы через сток
#  прошла вся масса коробки.  У коагуляции это НЕ M_sys/m_sink, как у дробления:
#  масса уходит кусками по m_sink, но чтобы довести один кусок снизу доверху,
#  нужно порядка R = m_sink/m_inj слияний.  Отсюда
#      промывка = (M_sys/m_sink) * R = M_sys/m_inj  событий.
MBAR  = AN.steady_mean_mass(PR["alpha"], M_INJ, M_SINK)
M_SYS = N_SS * MBAR

if START == "state":
    #  ПЕРЕЗАПУСК.  Ничего не обнуляется: времена сдвигаются на t_end предыдущей
    #  ноги и уходят в минус, поэтому tau = t - t_anc остаётся ПОЛНЫМ возрастом
    #  через шов.  Для коагуляции это и есть смысл всей затеи.
    _prev = AN.load(STATE_FILE)
    STATE = BF.state_of(_prev, source=STATE_FILE)
    IC    = {"state": STATE}
    N0    = int(STATE["mass"].size)
    _sp = AN.spectrum(_prev); _pl = AN.spectrum_plateau(_prev, spec=_sp)
    print("ПЕРЕЗАПУСК с %s" % STATE_FILE)
    print("  частиц %.4g | <m> = %.4g | масса %.4g | t уже накоплено %.4g"
          % (N0, STATE["mass"].mean(), STATE["mass"].sum(),
             STATE["t_origin"] + STATE["t_end"]))
    print("  спектр состояния: alpha = %+.4f +- %.4f на %.2f декадах"
          % (_pl["alpha"], _pl["scatter"], _pl["decades"]))
    _ss = AN.stationarity(_prev)
    print(_ss["report"])
    if not _ss["ok"]:
        print("  !! состояние НЕ стационарно.  Мерить с него можно, но это будет")
        print("     измерение переходного режима.  Лучше добить ещё один прогон.")
    del _prev
else:
    N0 = int(N_SS)
    IC = {"m": M_INJ, "N": N0}
    print("ХОЛОДНЫЙ СТАРТ: N0 = %d мономеров при m_inj" % N0)

print("population: N_ss = %.3g, <m> = %.3g  ->  масса коробки %.3g" % (N_SS, MBAR, M_SYS))

# --- injection rate: the one constant that does NOT survive a change of kernel ------
# Number balance in steady state:  q = accepted event rate = <K> N^2 / (2V).
# The original notebook wrote q = N^2/2, which is that formula with <K> = 1 -- true for
# the CONSTANT kernel and for nothing else.  N is dominated by the injection scale
# (alpha < -1), so most pairs are (m_inj, m_inj) and <K> is of order K(m_inj, m_inj);
# heavier partners push it up by roughly a factor two, so `live` settles somewhat below
# N_SS.  That is fine.  What matters is that panel (a) shows a PLATEAU and M_out/M_in
# heads for 1 -- retune K_TYP by hand if it drifts.
K_TYP  = KERNEL(M_INJ, M_INJ)                  # 1.0 для постоянного ядра
q3     = 0.5 * K_TYP * N_SS**2
t_c    = 1.0 / (K_TYP * N_SS)                  # collision time per particle: 1/(K n), V = 1
print("K(m_inj,m_inj) = %.4g   ->   q = %.3e,   t_c = %.3e" % (K_TYP, q3, t_c))

#  СКОРОСТЬ -- замеряется, а не угадывается.  Короткий пробный прогон без
#  поколений и без снимков: 200 тысяч событий стоят секунды, а вся арифметика
#  бюджета ниже опирается на них.
_t0  = time.time()
_cal = BF.simulate(process="coagulation", system="open", kernel=KERNEL, edges=EDGES,
                   ic={"m": M_INJ, "N": int(N_SS)}, injection_rate=q3,
                   injection_mass=M_INJ, sink_mass=M_SINK,
                   snapshot_mode="events", snapshot_stride=1e12,
                   max_events=200_000, track_generations=False,
                   rng=np.random.default_rng(0), verbose=False)
RATE = float(_cal["events"][-1]) / max(time.time() - _t0, 1e-9)
print("probe   : %.0f событий при %.3g ev/s, acceptance %.3f"
      % (_cal["events"][-1], RATE, _cal["acceptance"]))
del _cal

#  ДВА РЕЖИМА, и различает их SAVE_STATE -- отдельного переключателя не надо.
#    SAVE_STATE=True  -- прогон СТРОИТ стационар.  Длина в промывках, потому что
#                        именно в них считается сходимость и в них же отчитывается
#                        AN.stationarity.  Сток не тормозит.
#    SAVE_STATE=False -- прогон МЕРИТ.  Длина задаётся набором на стоке; промывка
#                        входит в неё, только если стартуем не с состояния.
_flush = M_SYS / M_INJ                 # событий на одну промывку коробки
_meas  = N_STOP * R                    # событий на N_STOP поглощений
CEIL   = int(RATE * 3600.0 * HOURS)
if SAVE_STATE:
    _need     = FLUSHES * _flush
    STOP_SINK = 10**15                 # строим: тормозит только бюджет событий
else:
    _need     = 1.2 * (_meas if START == "state" else _flush + _meas)
    STOP_SINK = N_STOP
MAX_EV = int(min(CEIL, _need))
E_EXP  = MAX_EV
print("cost    : промывка %.3g событий + %.3g на сток = %.3g, это %.2f ч"
      % (_flush, _meas, _flush + _meas, (_flush + _meas) / RATE / 3600.0))
print("          бюджет прогона %.3g событий = %.2f промывки = %.2f ч, упрётся в %s"
      % (MAX_EV, MAX_EV / _flush, MAX_EV / RATE / 3600.0,
         ("промывки" if SAVE_STATE else "сток") if _need <= CEIL
         else "ЧАСЫ -- подними HOURS"))
if START == "state":
    print("          промывка (%.3g) в бюджет НЕ входит: она оплачена прогонами до этого"
          % _flush)
print("brakes  : stop_sink_events = %.3g | max_events = %.2e" % (STOP_SINK, MAX_EV))
print("cadence : стрид %.2e, около 200 снимков" % max(int(E_EXP)//200, 1))

_t = time.time()
r3 = BF.simulate(process="coagulation", system="open", kernel=KERNEL, edges=EDGES,
                 ic=IC, injection_rate=q3, injection_mass=M_INJ,
                 sink_mass=M_SINK,
                 track_generations=True, gen_max=GEN_MAX,
                 snapshot_mode="events", snapshot_stride=max(int(E_EXP)//200, 1),
                 # max_events=np.inf is only safe when SOMETHING ELSE can stop the run.
                 # In an OPEN COAGULATION run nothing else can: live sits on a plateau so
                 # `live < 2` never fires, stop_max/min_mass are None, and the stall guard
                 # stays quiet because events keep being accepted.  Without the line below
                 # this cell runs forever.  stop_sink_events is the v2 brake and, unlike
                 # max_events, it means the same thing at every kernel -- but keep a
                 # resource ceiling as well, so a mistuned q cannot cost an afternoon.
                 max_events=MAX_EV, stop_sink_events=STOP_SINK,
                 iso_age_edges=10.0**np.arange(-3, 3, AGE_STEP) * t_c,
                 iso_start_sink=N_OUT,          # the gate (v2 default is 10 anyway)
                 age_rule="mass_weighted",      # report this; try 'min' to test sensitivity
                 rng=np.random.default_rng(SEED), verbose=True)

print("\n%.1f мин | live %d -> %d   (must plateau)"
      % ((time.time() - _t) / 60, r3["live"][0], r3["live"][-1]))
print("sink absorptions = %d  (gate asks for %d) | M_out/M_in = %.3f  (must approach 1)"
      % (r3["sink_events"], N_OUT, r3["M_out"][-1]/max(r3["M_in"][-1],1)))
print("isochrones: %d snapshots, accumulation began at t = %.3g  (t*N = %.0f)"
      % (r3["iso_snapshots"], r3["iso_t_begin"], r3["iso_t_begin"]*N_SS))
print("run ended at t = %.3g  (t*N = %.0f) | stop = %s"
      % (r3["t"][-1], r3["t"][-1]*N_SS, r3["stop_reason"]))

# ---- СТАЦИОНАРНОСТЬ и ЧЕКПОЙНТ --------------------------------------------
#  Настоящий вердикт: <m> и alpha, сравненные между половинами хвоста.  Оба
#  привычных критерия -- M_out/M_in и dn_out/dev -- садятся на единицу задолго до
#  того, как садится форма: число живёт у стока, форма наверху.
ST3 = AN.stationarity(r3)
print(ST3["report"])

if SAVE_STATE:
    #  СОХРАНЯЕТСЯ ВСЕГДА, стационарен прогон или нет.  Нестационарный чекпойнт --
    #  это ТОЧКА ПРОДОЛЖЕНИЯ, и весь итеративный порядок работы на ней держится:
    #  прогон, чекпойнт, проверка, ещё прогон с того же чекпойнта.
    _p = AN.add_last_run(r3, os.path.basename(STATE_FILE).replace(".npz", ""),
                         drop_particles=False,
                         runs_dir=os.path.dirname(STATE_FILE) or "runs",
                         analysis=dict(checkpoint=True, flushes=ST3["flushes"],
                                       stationary=bool(ST3["ok"]),
                                       drift_mbar=ST3["mbar"][2],
                                       drift_alpha=ST3["alpha"][2]))
    print("\nЧЕКПОЙНТ СОХРАНЁН -> %s" % _p)
    if ST3["ok"]:
        print("  состояние СТАЦИОНАРНО: годится как старт измерительного прогона.")
        print("  дальше: START='state', SAVE_STATE=False -- и мерить.")
    else:
        print("  состояние НЕ стационарно: это точка продолжения, а не старт измерения.")
        if np.isfinite(ST3.get("flushes_left", np.nan)):
            print("  дальше: START='state', SAVE_STATE=True -- ещё около %.1f промывки"
                  % ST3["flushes_left"])
            print("          = %.3g событий = %.1f ч при этой скорости"
                  % (ST3["flushes_left"] * _flush,
                     ST3["flushes_left"] * _flush / RATE / 3600.0))
        else:
            print("  дальше: START='state', SAVE_STATE=True -- ещё прогон;")
            print("          срок назвать нельзя, дрейф ещё не начал затухать.")

# ---------------- the spectrum -------------------------------------------------
F3, NSS3 = steady_spectrum(r3)
print("estimator: mean of %d snapshots taken after the sink gate" % NSS3)

# THE measurement: the longest plateau in the local slope.  Everything downstream --
# the model on panel (b), the summary row -- reads this and nothing else.
ir3 = BF.find_inertial_range(r3["centers"], F3)

# a-priori band, kept only as an independent cross-check.  NOTE the upper scale is M_SINK,
# per guard_band's own docstring (open system: [m_inj, m_sink]).
gb3 = BF.guard_band(M_INJ, M_SINK, 0.5)
f3  = BF.fit_powerlaw(r3["centers"], F3, *gb3)
pr3 = PR

# ---------------- the growth law, from the isochrones --------------------------
tau3, mbar3, n3 = BF.iso_mean_mass(r3["iso_counts"], r3["centers"], r3["iso_age_edges"])
# The window was hard-coded (3, 300) for the constant kernel.  Tie it to the guard band
# instead: fit the growth law only where the spectrum itself is a clean power law.  At
# b = 6 a 2-decade mass window is a THIRD of a decade in tau and catches two age bins.
k3 = np.isfinite(mbar3) & (n3 > 1e3) & (mbar3 > gb3[0]) & (mbar3 < gb3[1])
b3 = np.polyfit(np.log(tau3[k3]), np.log(mbar3[k3]), 1)[0] if k3.sum() > 3 else np.nan

print("\nspectrum, AUTO PLATEAU [%.3g,%.3g] : alpha = %+.3f +- %.3f  over %.2f decades   <== the answer"
      % (ir3["m_lo"], ir3["m_hi"], ir3["alpha"], ir3["scatter"], ir3["decades"]))
print("spectrum, guard band   [%.3g,%.3g] : alpha = %+.3f   R2 = %.3f   (cross-check)"
      % (gb3[0], gb3[1], f3["alpha"], f3["r2"]))
print("theory                              : alpha = %+.3f   b = %.3f" % (pr3["alpha"], pr3["b"]))
print("isochrone growth                    : b = %.3f   [%d age bins]" % (b3, k3.sum()))
if k3.sum() <= 3:
    print("  !! too few usable age bins: %d iso snapshots, %d sink absorptions."
          "  Raise N_STOP (cost ~ N*sqrt(m_sink) + n_out*m_sink) or lower M_SINK."
          % (r3["iso_snapshots"], r3["sink_events"]))

RESULTS["3 open coag"] = dict(b=b3, b_th=pr3["b"], alpha=ir3["alpha"], alpha_th=pr3["alpha"],
                              scatter=ir3["scatter"], decades=ir3["decades"],
                              alpha_gb=f3["alpha"])


# ---------------- persist, for the analysis notebooks in runs/ -----------------
KTAG      = KERNEL.__name__.replace("kernel_", "")
RUN_NAME3 = "open_coagulation_conservative_%s" % KTAG
add_last_run(r3, RUN_NAME3, analysis=dict(
    alpha_plateau=ir3["alpha"], alpha_scatter=ir3["scatter"],
    plateau_decades=ir3["decades"], plateau_m_lo=ir3["m_lo"], plateau_m_hi=ir3["m_hi"],
    alpha_guard=f3["alpha"], guard_r2=f3["r2"], guard_lo=gb3[0], guard_hi=gb3[1],
    alpha_theory=pr3["alpha"], b_theory=pr3["b"], b_iso=b3,
    iso_age_bins_used=k3.sum(), lam=LAM, N_ss=N_SS, m_inj=M_INJ, m_sink=M_SINK,
    q=q3, K_typ=K_TYP, t_c=t_c, age_step=AGE_STEP,
    n_out_gate=N_OUT, n_out_stop=N_STOP,
    start=START, checkpoint=False, stationary=bool(ST3["ok"]),
    flushes=ST3["flushes"]))
if SAVE_STATE:
    print("\nЭТО СТРОЯЩИЙ ПРОГОН.  Файл %s.npz перезаписан, но мерить по нему рано:"
          % RUN_NAME3)
    print("гони измерительный (START='state', SAVE_STATE=False), он его и заменит.")


In [ ]:
N_ISO_SHOW = 1000      # <<-- how many isochrones to draw on panel (b).  The line below prints
                    #      how many are available first, so you always know what you are seeing.
                    #      NOTE: AGE_STEP is now 0.05 dex (b = 6), so there are ~119 age bins
                    #      instead of 39 and "100" really does draw ~70 curves.  Drop to ~12
                    #      if panel (b) turns into a hairball.

idx3, tau3c, navail3 = pick_isochrones(r3, N_ISO_SHOW)
print("isochrones: %d of %d age bins carry statistics -> drawing %d"
      % (navail3, len(tau3c), len(idx3)))

fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.2))

# ---- (a) steady-state check, with the sink counter on the right axis ----------
ax[0].plot(r3["t"], r3["live"], ".-", ms=3, lw=.8, color="C0")
ax[0].set_xlabel("t"); ax[0].set_ylabel("live particles", color="C0")
ax[0].set_ylim(0, 1.2*np.max(r3["live"]))      # start at zero: no offset trick on this axis
ax[0].set_title("(a) steady-state check")
axr = ax[0].twinx(); axr.grid(False)
axr.plot(r3["t"], r3["n_out"], "-", lw=1.3, color="C1")
axr.set_ylabel(r"absorbed at sink  $n_{\rm out}$", color="C1")
if np.isfinite(r3["iso_t_begin"]):
    ax[0].axvline(r3["iso_t_begin"], ls="--", lw=1, color="C3")
    ax[0].text(r3["iso_t_begin"], ax[0].get_ylim()[1], " isochrones start\n (%d sink hits)" % N_OUT,
               fontsize=6, va="top", color="C3")

# ---- (b) COMPENSATED spectrum: m^2 dN/dm = mass per logarithmic mass interval --
c = r3["centers"]
a = ax[1]
draw_isochrones(a, r3, idx3, tau3c, tau_scale=N_SS)
if len(idx3):
    # completeness check: the isochrones must add up to the steady state
    iso_sum = r3["iso_dndm"].sum(axis=0) / max(r3["iso_snapshots"], 1)
    a.loglog(c, np.where(iso_sum > 0, iso_sum*c**2, np.nan), "-", lw=3.5, alpha=.35,
             color="0.4", zorder=2, label="sum of isochrones")
a.loglog(c, np.where(F3 > 0, F3*c**2, np.nan), "o", ms=3, color="k", zorder=4,
         label="steady state")

xs  = np.logspace(np.log10(ir3["m_lo"]), np.log10(ir3["m_hi"]), 30)
A_m = anchor_amplitude(c, F3, ir3["m_lo"], ir3["m_hi"], ir3["alpha"])
A_t = anchor_amplitude(c, F3, ir3["m_lo"], ir3["m_hi"], pr3["alpha"])
a.loglog(xs, A_m * xs**ir3["alpha"] * xs**2, "-",  lw=2.2, color="C3", zorder=5,
         label=r"plateau $\alpha=%.2f$" % ir3["alpha"])
a.loglog(xs, A_t * xs**pr3["alpha"] * xs**2, "--", lw=1.4, color="C1", zorder=5,
         label=r"theory $%.2f$" % pr3["alpha"])
compensated_ylim(a, c, F3)
a.set_xlim(EDGES[0], 3 * M_SINK); a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
a.legend(fontsize=6, loc="lower left"); a.set_title(r"(b) $m^2\,dN/dm$ + isochrones")

# ---- (c) isochrone growth law -------------------------------------------------
if k3.sum() > 3:
    ax[2].loglog(tau3[k3], mbar3[k3], "o", ms=3)
    ax[2].loglog(tau3[k3], mbar3[k3][0]*(tau3[k3]/tau3[k3][0])**b3, "-", lw=1,
                 label=r"$\langle m\rangle\propto\tau^{%.2f}$ (theory %.0f)" % (b3, pr3["b"]))
    ax[2].legend(fontsize=7)
else:
    ax[2].set_xscale("log"); ax[2].set_yscale("log")
    ax[2].text(.5, .5, "no usable isochrones\n%d iso snapshots, %d sink hits"
               % (r3["iso_snapshots"], r3["sink_events"]),
               ha="center", va="center", fontsize=7, color="C3", transform=ax[2].transAxes)
ax[2].set_xlabel(r"age $\tau$"); ax[2].set_ylabel(r"$\langle m\rangle$")
ax[2].set_title("(c) isochrone growth law")
fig.tight_layout()


---
## Summary

One row per run. `plateau` is the mean local slope over the longest flat stretch of
$\Gamma(m)$, with its scatter and its width in decades; `guard` is the a-priori band
fit kept as an independent cross-check, and the two are expected to agree. `b` comes
from the isochrones, read as $\langle m\rangle\propto\tau^{\,b}$ for coagulation and
as $\langle m\rangle\propto(\tau_*-\tau)^{\,b}$ for fragmentation.

Read `dec` before believing `plateau`. Плато уже одной декады — не степенной закон,
как бы туго ни выглядела ошибка рядом с ним, и на строящем прогоне (`SAVE_STATE=True`)
оно узко именно потому, что каскад ещё строится. Строка из строящего прогона —
диагноз, а не измерение.

Every run above has been written into `runs/`. The analysis notebooks there reload
those files and rebuild each figure separately, so nothing below needs to be re-run
to change a plot.


In [ ]:
hdr = ("%-22s | %7s %7s | %8s %7s %6s | %8s %8s" %
       ("case", "b", "b_th", "plateau", "+-", "dec", "guard", "theory"))
print(hdr); print("-" * len(hdr))
for name, d in RESULTS.items():
    print("%-22s | %7.3f %7.3f | %+8.3f %7.3f %6.2f | %+8.3f %+8.3f" %
          (name, d["b"], d["b_th"], d["alpha"], d["scatter"], d["decades"],
           d["alpha_gb"], d["alpha_th"]))
print("\nplateau = find_inertial_range (longest flat run in the local slope) -- THE answer.")
print("guard   = the a-priori band fit, kept as an independent cross-check.")
print("dec     = plateau width in decades; below ~1 the index is not a measurement.")
print("b = nan  = the age axis was degenerate, see the note in the case-4 cell.")
